Prodzynski et al released their data as a .R object. If I was working in R/Seurat, this would be super easy and convenient to work with. Unfortunately, I am not, so I have to convert it to actually use it in python. This script uses a mix of R and python to do so, before the processing

In [ ]:
library(Seurat)
library(Matrix)

Loading required package: SeuratObject
Loading required package: sp
‘SeuratObject’ was built under R 4.5.1 but the current version is
4.5.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed
‘SeuratObject’ was built with package ‘Matrix’ 1.7.3 but the current
version is 1.7.4; it is recomended that you reinstall ‘SeuratObject’ as
the ABI for ‘Matrix’ may have changed

Attaching package: ‘SeuratObject’

The following objects are masked from ‘package:base’:

    intersect, t



In [ ]:
seurat_obj <- readRDS("../rawFiles/Prodzynski/GSE263372_combined_seurat.rds")


Writing the Seurat object into a matrix object file. additionally putting the metadata and gene list into CSVs.

In [ ]:
prefix <- "../rawFiles/Prodzynski/Converted_"
writeMM(obj = GetAssayData(seurat_obj, assay = "RNA", layer = "counts"), 
        file = paste0(prefix,"matrix.mtx"))
write.csv(seurat_obj@meta.data, file = paste0(prefix,"metadata.csv"), row.names = TRUE)
write.csv(rownames(seurat_obj), file = paste0(prefix,"genes.csv"), row.names = FALSE)


Now we can switch the kernal to Python and run the rest.

In [1]:
import scanpy as sc
import pandas as pd

Grab the created matrix and metadata and variable names, assigning them appropriately within an anndata object

In [10]:
prefix = "../rawFiles/Prodzynski/Converted_"
adata = sc.read_mtx(prefix + "matrix.mtx").T
genes = pd.read_csv(prefix + "genes.csv")
print(genes.x)
adata.var_names = genes.x
metadata = pd.read_csv(prefix + "metadata.csv", index_col=0)
adata.obs = metadata

0        AL627309.1
1        AL627309.5
2        AL627309.4
3        AP006222.2
4        AL669831.2
            ...    
26787    AF127936.1
26788    AP000431.1
26789    AP000275.2
26790     KRTAP10-1
26791     KRTAP10-3
Name: x, Length: 26792, dtype: object


Now save to a h5ad file, and we don't have to run or touch this again

In [ ]:
adata.write("../rawFiles/Prodzynski/converted_data.h5ad")